# Fake Job Postings Binomial Model

In [1]:
#download data
import pandas as pd
df=pd.read_csv('fake_job_postings.csv')

### Part 1: Grouping categorical variables

Many categorical variables have values that are very similar to each other (ex. Vocational vs Vocational - Degree), and others have values with so few observations, that it makes more sense to group them with another, similar category to create less dummy variables for the final regression (ex. grouping "Legal Services" and "Law Practice" roles together to increase the sample size for this group). Below, we do this with several categorical variables.

Clean salary data by the minimum end of the spectrum and bucket it into groups (including a missing group).

In [2]:
#default is missing, will override for non-missing values
df['salary_min']='missing'

#function to get first half of salary range
def slice_string(x):
    if type(x)==float:
        return 'missing'
    else:
        divider=x.find('-')
        xnew=x[:divider]
        return xnew

#take the minimum of the salary range
df['salary_min']=df['salary_range'].apply(slice_string)

In [3]:
#clean minimum end of salary range
def clean_minimum(x):
    if x=='Dec' or x=='Jun' or x=='missing' or x=='Oct' or len(x)==1:
        return 'missing' #nonsensical values should be missing
    elif len(x)==2 or len(x)==3:
        return int(x+'00') #assume a value of 10 means 10k and a value of 100 means 100k
    else:
        return int(x)

df['salary_min2']=df['salary_min'].apply(clean_minimum)

#### Creating Buckets

In [4]:
#apply buckets
def bucket_minimum(x):
    if x=='missing':
        return 'missing'
    elif x<50000:
        return 'low'
    elif x<100000:
        return 'medium'
    else:
        return 'high'

df['salary_bucket']=df['salary_min2'].apply(bucket_minimum)

In [5]:
# country (only keeping US values)
df['country']=df['location'].str[:2]
df = df[df['country'] == "US"]
df = df.drop(columns=['country'])

In [6]:
#bucket education into slightly broader groups
def bucket_education(x):
    if x == "Bachelor's Degree":
        return x
    elif x=="Master's Degree" or x=="Doctorate":
        return "Above Bachelor's"
    elif x=="Vocational" or x=="Vocational - Degree" or x=="Vocational - HS Diploma" or x=="Associate Degree" or x=="Certification" or x=="Professional":
        return "Vocational, Associate's Degree, or Certificate"
    elif x=="High School or equivalent" or x=="Some College Coursework Completed" or x=="Some High School Coursework":
        return "Below college degree"
    else:
        return "Missing"


df['bucketed_education']=df['required_education'].apply(bucket_education)

In [7]:
# work experience
def bucket_experience(x):
    if x == "Internship" or x=="Entry level" or x=="Associate":
        return "Work experience"
    elif x=="Mid-Senior level" or x=="Executive" or x=="Director":
        return "Leadership experience"
    else:
        return "No experience required"

df['bucketed_experience']=df['required_experience'].apply(bucket_experience)

In [8]:
# employment type
def bucket_type(x):
    if x == "Full-time" or x=="Part-time" or x=="Other":
        return x
    elif x=="Contract" or x=="Temporary":
        return "Temporary"
    else:
        return "missing"

df['bucketed_employment_type']=df['employment_type'].apply(bucket_type)

In [9]:
# industry
def bucket_industry(x):
    if x in ['Marketing and Advertising', 'Staffing and Recruiting', 'Public Relations and Communications']:
        return 'Marketing, PR, Recruiting'
    elif x in ['Management Consulting', 'Human Resources', 'Program Development', 'Executive Office']:
        return "Consulting/development"
    elif x in ['Computer Software', 'Online Media', 'Information Technology and Services', 'Internet', 'Consumer Electronics', 'Information Services', 'Computer Networking', 'Biotechnology', 'Wireless', 'Computer Hardware', 'Computer & Network Security', 'Security and Investigations', 'Nanotechnology']:
        return "Technology"
    elif x in ['Financial Services', 'Banking', 'Insurance', 'Venture Capital & Private Equity', 'Market Research', 'Accounting', 'Investment Management', 'Investment Banking', 'Capital Markets']:
        return "Finance"
    elif x in ['Oil & Energy', 'Food Production', 'Farming', 'Machinery', 'Mechanical or Industrial Engineering', 'Electrical/Electronic Manufacturing', 'Chemicals', 'Aviation & Aerospace', 'Airlines/Aviation', 'Transportation/Trucking/Railroad', 'Semiconductors', 'Maritime', 'Mining & Metals', 'Textiles', 'Civil Engineering', 'Plastics', 'Fishery', 'Furniture', 'Shipbuilding', 'Ranching', 'Industrial Automation']:
        return "Engineering/Manufacturing/Trades"
    elif x in ['Food & Beverages', 'Health, Wellness and Fitness', 'Cosmetics','Leisure, Travel & Tourism', 'Luxury Goods & Jewelry', 'Hospitality', 'Gambling & Casinos', 'Apparel & Fashion', 'Sports', 'Computer Games', 'Restaurants', 'Entertainment', 'Sporting Goods','Professional Training & Coaching', 'Wine and Spirits']:
        return 'Leisure/Luxury'
    elif x in ['Construction', 'Real Estate', 'Commercial Real Estate']:
        return 'Real Estate'
    elif x in ['Philanthropy', 'Civic & Social Organization', 'Religious Institutions','Nonprofit Organization Management', 'Libraries', 'Government Administration', 'Fund-Raising', 'Government Relations', 'Individual & Family Services', 'Public Policy', 'Museums and Institutions']:
        return 'Govt/Nonprofit'
    elif x in ['Education Management', 'E-Learning', 'Primary/Secondary Education', 'Research', 'Higher Education']:
        return "Education/Research"
    elif x in ['Legal Services', 'Law Practice', 'Law Enforcement', 'Public Safety', 'Alternative Dispute Resolution']:
        return "Legal System"
    elif x in ['Pharmaceuticals', 'Medical Practice', 'Medical Devices', 'Mental Health Care', 'Veterinary']:
        return "Healthcare"
    elif x in ['Music', 'Design', 'Media Production', 'Publishing', 'Writing and Editing', 'Graphic Design', 'Broadcast Media', 'Animation', 'Motion Pictures and Film', 'Photography', 'Performing Arts']:
        return "Art/Media"
    elif x in ['Logistics and Supply Chain', 'Business Supplies and Equipment', 'Warehousing', 'Outsourcing/Offshoring', 'International Trade and Development', 'Import and Export']:
        return "Operations"
    elif x in ['Events Services', 'Facilities Services', 'Telecommunications', 'Building Materials', 'Utilities', 'Environmental Services', 'Printing', 'Wholesale', 'Architecture & Planning', 'Renewables & Environment', 'Packaging and Containers', 'Package/Freight Delivery', 'Translation and Localization', 'Military', 'Defense & Space']:
        return "Other"
    else:
        return "missing"

df['bucketed_industry']=df['industry'].apply(bucket_industry)

In [10]:
import statsmodels.api as sm
dummies_only=pd.get_dummies(df[['salary_bucket','bucketed_education', 'bucketed_experience', 'bucketed_employment_type', 'bucketed_industry']], columns=['salary_bucket','bucketed_education', 'bucketed_experience', 'bucketed_employment_type', 'bucketed_industry'], drop_first=True).astype(int)
X=sm.add_constant(dummies_only)
y = df['fraudulent']  # 1 = fake, 0 = real
binomial_model = sm.GLM(y, X, family=sm.families.Binomial()).fit()
print(binomial_model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:             fraudulent   No. Observations:                10656
Model:                            GLM   Df Residuals:                    10628
Model Family:                Binomial   Df Model:                           27
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -2361.3
Date:                Tue, 09 Dec 2025   Deviance:                       4722.6
Time:                        17:27:48   Pearson chi2:                 1.02e+04
No. Iterations:                     9   Pseudo R-squ. (CS):            0.05477
Covariance Type:            nonrobust                                         
                                                                        coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------

In [11]:
# Create all dummy variables without automatically dropping the first category
dummies_true = pd.get_dummies(df[['salary_bucket', 'bucketed_education', 'bucketed_experience', 'bucketed_employment_type', 'bucketed_industry']],
                              columns=['salary_bucket','bucketed_education', 'bucketed_experience', 'bucketed_employment_type', 'bucketed_industry'],
                              drop_first=False).astype(int)

# Identify the 'missing' related dummy columns to be excluded from the model
missing_cols_to_drop = [
    'salary_bucket_missing',
    'country_Other', # 'Other' is the bucketed 'missing' for country
    'bucketed_education_Missing',
    'bucketed_experience_No experience required', # 'No experience required' is the bucketed 'missing' for experience
    'bucketed_employment_type_missing',
    'bucketed_function_missing',
    'bucketed_industry_missing'
]

# Drop these identified columns from the dummy variables DataFrame
# Using errors='ignore' ensures that the code doesn't fail if a column doesn't exist
existing_missing_cols = [col for col in missing_cols_to_drop if col in dummies_true.columns]
dummies_true = dummies_true.drop(columns=existing_missing_cols, errors='ignore')

# proceed with regression as before
X=sm.add_constant(dummies_true)
y = df['fraudulent']  # 1 = fake, 0 = real
binomial_model = sm.GLM(y, X, family=sm.families.Binomial()).fit()
print(binomial_model.summary())


                 Generalized Linear Model Regression Results                  
Dep. Variable:             fraudulent   No. Observations:                10656
Model:                            GLM   Df Residuals:                    10628
Model Family:                Binomial   Df Model:                           27
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -2361.3
Date:                Tue, 09 Dec 2025   Deviance:                       4722.6
Time:                        17:27:48   Pearson chi2:                 1.02e+04
No. Iterations:                     9   Pseudo R-squ. (CS):            0.05477
Covariance Type:            nonrobust                                         
                                                                        coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------

### Part 2: Turning qualitative variables into quantitative

We can use the length of company profile, description, requirements, and benefits, to see if more robust/lengthy job postings are indicative of fraud or not.

In [12]:
def get_length(x):
    if type(x)==float:
        return 0 #dealing with missing values
    else:
        return len(x)

Get length and normalized length (to avoid feature dominance), starting with company profile.

In [13]:
import statistics
# company profile
df['profile_length']=df['company_profile'].apply(get_length)
mean=df['profile_length'].mean()
std_dev = statistics.stdev(df['profile_length'])
df['normalized_profile_length']=(df['profile_length']-mean)/std_dev

# description
df['description_length']=df['description'].apply(get_length)
mean=df['description_length'].mean()
std_dev = statistics.stdev(df['description_length'])
df['normalized_description_length']=(df['description_length']-mean)/std_dev

# requirements
df['requirements_length']=df['requirements'].apply(get_length)
mean=df['requirements_length'].mean()
std_dev = statistics.stdev(df['requirements_length'])
df['normalized_requirements_length']=(df['requirements_length']-mean)/std_dev

# benefits
df['benefits_length']=df['benefits'].apply(get_length)
mean=df['benefits_length'].mean()
std_dev = statistics.stdev(df['benefits_length'])
df['normalized_benefits_length']=(df['benefits_length']-mean)/std_dev

### Part 3: Restrict to useful variables

Remember, X is already our data frame of relevant dummies plus the constant. So now we will just take all the other variables we need and merge the two data sets together.

In [14]:
df_tokeep=df[['has_company_logo', 'has_questions', 'normalized_profile_length',
              'normalized_description_length', 'normalized_requirements_length', 'normalized_benefits_length', 'fraudulent']]

df_final = pd.concat([X, df_tokeep], axis=1)

In [15]:
df_final.head(5)

,const,salary_bucket_high,salary_bucket_low,salary_bucket_medium,bucketed_education_Above Bachelor's,bucketed_education_Bachelor's Degree,bucketed_education_Below college degree,"bucketed_education_Vocational, Associate's Degree, or Certificate",bucketed_experience_Leadership experience,bucketed_experience_Work experience,...,bucketed_industry_Other,bucketed_industry_Real Estate,bucketed_industry_Technology,has_company_logo,has_questions,normalized_profile_length,normalized_description_length,normalized_requirements_length,normalized_benefits_length,fraudulent
0,1.0,0,0,0,0,0,0,0,0,1,...,0,0,0,1,0,0.570247,-0.394292,0.416135,-0.581944,0
2,1.0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0.559081,-0.970145,1.223449,-0.581944,0
3,1.0,0,0,0,0,1,0,0,1,0,...,0,0,1,1,0,0.065934,1.380382,1.327720,1.944687,0
4,1.0,0,0,0,0,1,0,0,1,0,...,0,0,0,1,1,1.952921,0.249616,0.266047,-0.514093,0
5,1.0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,-1.076680,2.236832,-0.929916,-0.581944,0


### Part 4: Initial regression attempts

In [16]:
#intercept only model for reference
intercept_model = sm.GLM(df_final[['fraudulent']], df_final['const'], family=sm.families.Binomial()).fit()
print(intercept_model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:             fraudulent   No. Observations:                10656
Model:                            GLM   Df Residuals:                    10655
Model Family:                Binomial   Df Model:                            0
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -2661.4
Date:                Tue, 09 Dec 2025   Deviance:                       5322.8
Time:                        17:27:49   Pearson chi2:                 1.07e+04
No. Iterations:                     6   Pseudo R-squ. (CS):         -2.220e-16
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -2.6099      0.038    -68.057      0.0

In [17]:
#all variables
y = df_final['fraudulent'] # target
X = df_final.drop(columns=['fraudulent'])
X = sm.add_constant(X)

logit_model = sm.GLM(y, X, family=sm.families.Binomial()).fit()
print(logit_model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:             fraudulent   No. Observations:                10656
Model:                            GLM   Df Residuals:                    10622
Model Family:                Binomial   Df Model:                           33
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -2024.6
Date:                Tue, 09 Dec 2025   Deviance:                       4049.1
Time:                        17:27:49   Pearson chi2:                 1.19e+04
No. Iterations:                     9   Pseudo R-squ. (CS):             0.1127
Covariance Type:            nonrobust                                         
                                                                        coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------

##### Likelihood ratio

In [18]:
from scipy.stats import chi2

null_deviance = 5322.8
residual_deviance = 4049.1
df_diff = 33  # number of predictors added

# Likelihood ratio
lr_stat = null_deviance - residual_deviance
print("Likelihood Ratio:", lr_stat)

# p-value
p_value = chi2.sf(lr_stat, df_diff)
print("p-value:", p_value)

Likelihood Ratio: 1273.7000000000003
p-value: 1.5053979402372143e-246


Our model is significantly different from the null model.

### Part 5: Interaction terms

In [19]:
# education x experience
df_final['abovebachelor_x_leadershipexperience'] = df_final["bucketed_education_Above Bachelor's"] * df_final['bucketed_experience_Leadership experience']
df_final['bachelors_x_leadershipexperience'] = df_final["bucketed_education_Bachelor's Degree"] * df_final['bucketed_experience_Leadership experience']
df_final['belowcollege_x_leadershipexperience'] = df_final["bucketed_education_Below college degree"] * df_final['bucketed_experience_Leadership experience']
df_final['vocational_x_leadershipexperience'] = df_final["bucketed_education_Vocational, Associate's Degree, or Certificate"] * df_final['bucketed_experience_Leadership experience']

df_final['abovebachelor_x_workexperience'] = df_final["bucketed_education_Above Bachelor's"] * df_final['bucketed_experience_Work experience']
df_final['bachelors_x_workexperience'] = df_final["bucketed_education_Bachelor's Degree"] * df_final['bucketed_experience_Work experience']
df_final['belowcollege_x_workexperience'] = df_final["bucketed_education_Below college degree"] * df_final['bucketed_experience_Work experience']
df_final['vocational_x_workexperience'] = df_final["bucketed_education_Vocational, Associate's Degree, or Certificate"] * df_final['bucketed_experience_Work experience']


In [20]:
# length x full-time
df_final['FT_x_profile'] = df_final['bucketed_employment_type_Full-time'] * df_final['normalized_profile_length']
df_final['FT_x_benefits'] = df_final['bucketed_employment_type_Full-time'] * df_final['normalized_benefits_length']
df_final['FT_x_desc'] = df_final['bucketed_employment_type_Full-time'] * df_final['normalized_description_length']
df_final['FT_x_req'] = df_final['bucketed_employment_type_Full-time'] * df_final['normalized_requirements_length']

In [21]:
# length x part-time
df_final['PT_x_profile'] = df_final['bucketed_employment_type_Part-time'] * df_final['normalized_profile_length']
df_final['PT_x_benefits'] = df_final['bucketed_employment_type_Part-time'] * df_final['normalized_benefits_length']
df_final['PT_x_desc'] = df_final['bucketed_employment_type_Part-time'] * df_final['normalized_description_length']
df_final['PT_x_req'] = df_final['bucketed_employment_type_Part-time'] * df_final['normalized_requirements_length']

In [22]:
# length x temporary
df_final['temp_x_profile'] = df_final['bucketed_employment_type_Temporary'] * df_final['normalized_profile_length']
df_final['temp_x_benefits'] = df_final['bucketed_employment_type_Temporary'] * df_final['normalized_benefits_length']
df_final['temp_x_desc'] = df_final['bucketed_employment_type_Temporary'] * df_final['normalized_description_length']
df_final['temp_x_req'] = df_final['bucketed_employment_type_Temporary'] * df_final['normalized_requirements_length']

In [23]:
#remove outleirs
df_final=df_final[df_final['normalized_profile_length']<3]
df_final=df_final[df_final['normalized_description_length']<3]
df_final=df_final[df_final['normalized_requirements_length']<3]
df_final=df_final[df_final['normalized_benefits_length']<3]

In [24]:
import statsmodels.api as sm

X = sm.add_constant(df_final.drop(columns=['fraudulent']))  # predictors
y = df_final['fraudulent']                                  # target

interaction_model = sm.GLM(y, X, family=sm.families.Binomial()).fit()
print(interaction_model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:             fraudulent   No. Observations:                 9955
Model:                            GLM   Df Residuals:                     9901
Model Family:                Binomial   Df Model:                           53
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -1766.3
Date:                Tue, 09 Dec 2025   Deviance:                       3532.6
Time:                        17:27:49   Pearson chi2:                 1.07e+04
No. Iterations:                     9   Pseudo R-squ. (CS):             0.1301
Covariance Type:            nonrobust                                         
                                                                        coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------

##### Checking for multicollinearity

In [25]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif = pd.DataFrame()
vif["feature"] = X.columns
vif["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(vif.sort_values(by="VIF", ascending=False).head(15))

                                              feature        VIF
0                                               const  12.006250
40                      belowcollege_x_workexperience   8.704149
6             bucketed_education_Below college degree   7.445137
7   bucketed_education_Vocational, Associate's Deg...   6.725512
32                     normalized_requirements_length   6.455366
5                bucketed_education_Bachelor's Degree   6.261599
33                         normalized_benefits_length   6.113226
30                          normalized_profile_length   6.030411
35                   bachelors_x_leadershipexperience   5.736347
43                                      FT_x_benefits   5.601525
41                        vocational_x_workexperience   5.520212
31                      normalized_description_length   5.459797
45                                           FT_x_req   5.447235
44                                          FT_x_desc   4.465308
42                       

**Checking for joint significance**

In [26]:
salary_dummies = X.filter(regex="^salary").columns.tolist()
edu_dummies=X.filter(regex="^bucketed_education").columns.tolist()
exper_dummies=X.filter(regex="^bucketed_exper").columns.tolist()
employment_dummies=X.filter(regex="^bucketed_employ").columns.tolist()
industry_dummies=X.filter(regex="^bucketed_ind").columns.tolist()
length_vars=X.filter(regex="^normalized_").columns.tolist()
educ_exper=['abovebachelor_x_leadershipexperience', 'bachelors_x_leadershipexperience', 'belowcollege_x_leadershipexperience', 'vocational_x_leadershipexperience',
              'abovebachelor_x_workexperience','bachelors_x_workexperience','belowcollege_x_workexperience', 'vocational_x_workexperience']
type_length=['FT_x_profile', 'FT_x_benefits', 'FT_x_desc', 'FT_x_req','PT_x_profile', 'PT_x_benefits', 'PT_x_desc', 'PT_x_req', 'temp_x_profile', 'temp_x_benefits', 'temp_x_desc', 'temp_x_req']

In [27]:
full_model = interaction_model

for vars in [salary_dummies, edu_dummies, exper_dummies, employment_dummies, industry_dummies, length_vars, educ_exper, type_length, 'has_questions', 'has_company_logo']:
# Restricted model: excludes country dummies(testing their joint significance)
    restricted_model = sm.GLM(y, X.drop(columns=vars, errors='ignore'), family=sm.families.Binomial()).fit()

# Likelihood Ratio Test
    LR_stat = 2 * (full_model.llf - restricted_model.llf)  # Test statistic
    df_diff = full_model.df_model - restricted_model.df_model  # Degrees of freedom
    p_value = chi2.sf(LR_stat, df_diff)  # Chi-square survival function
    print(p_value)
    if p_value < 0.05:
        print("Result: Reject null hypothesis — variables are jointly significant.", vars)
    else:
        print("Result: Fail to reject null — variables are not jointly significant.", vars)


9.629648380655092e-21
Result: Reject null hypothesis — variables are jointly significant. ['salary_bucket_high', 'salary_bucket_low', 'salary_bucket_medium']
0.043477695015246905
Result: Reject null hypothesis — variables are jointly significant. ["bucketed_education_Above Bachelor's", "bucketed_education_Bachelor's Degree", 'bucketed_education_Below college degree', "bucketed_education_Vocational, Associate's Degree, or Certificate"]
2.353923985423967e-06
Result: Reject null hypothesis — variables are jointly significant. ['bucketed_experience_Leadership experience', 'bucketed_experience_Work experience']
1.0714445743455305e-05
Result: Reject null hypothesis — variables are jointly significant. ['bucketed_employment_type_Full-time', 'bucketed_employment_type_Other', 'bucketed_employment_type_Part-time', 'bucketed_employment_type_Temporary']
1.99715325983879e-34
Result: Reject null hypothesis — variables are jointly significant. ['bucketed_industry_Art/Media', 'bucketed_industry_Consul

### Part 6: Testing/Training

In [28]:
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve

# test/train split
X = df_final.drop(columns=['fraudulent']) # predictors
y = df_final['fraudulent'] # target

X_train, X_test, y_train, y_test = train_test_split( # 80/20 split
    X, y, test_size=0.2, random_state=42, stratify=y
)

# add constant
X_train = sm.add_constant(X_train)
X_test = sm.add_constant(X_test)

# train model
interaction_model = sm.GLM(y_train, X_train, family=sm.families.Binomial())
result = interaction_model.fit()
print(result.summary())

# test model
y_pred_prob = result.predict(X_test) # Probabilities
y_pred = (y_pred_prob > 0.5).astype(int) # Convert to classes


# ROC-AUC
auc = roc_auc_score(y_test, y_pred_prob)
print("ROC-AUC:", auc)

                 Generalized Linear Model Regression Results                  
Dep. Variable:             fraudulent   No. Observations:                 7964
Model:                            GLM   Df Residuals:                     7910
Model Family:                Binomial   Df Model:                           53
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -1412.4
Date:                Tue, 09 Dec 2025   Deviance:                       2824.8
Time:                        17:27:58   Pearson chi2:                 8.07e+03
No. Iterations:                     9   Pseudo R-squ. (CS):             0.1305
Covariance Type:            nonrobust                                         
                                                                        coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------

**Checking for multicollinearity**

In [29]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif = pd.DataFrame()
vif["feature"] = X.columns
vif["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(vif.sort_values(by="VIF", ascending=False).head(15))

                                              feature        VIF
0                                               const  12.006250
40                      belowcollege_x_workexperience   8.704149
6             bucketed_education_Below college degree   7.445137
7   bucketed_education_Vocational, Associate's Deg...   6.725512
32                     normalized_requirements_length   6.455366
5                bucketed_education_Bachelor's Degree   6.261599
33                         normalized_benefits_length   6.113226
30                          normalized_profile_length   6.030411
35                   bachelors_x_leadershipexperience   5.736347
43                                      FT_x_benefits   5.601525
41                        vocational_x_workexperience   5.520212
31                      normalized_description_length   5.459797
45                                           FT_x_req   5.447235
44                                          FT_x_desc   4.465308
42                       

In [30]:
#Identifying correlation
corr_matrix = X[['belowcollege_x_workexperience', 'normalized_benefits_length', 'bucketed_education_Below college degree', 'FT_x_benefits', "bucketed_education_Bachelor's Degree", "bucketed_education_Vocational, Associate's Degree, or Certificate", "bachelors_x_leadershipexperience", "normalized_profile_length"]].corr()
corr_matrix

,belowcollege_x_workexperience,normalized_benefits_length,bucketed_education_Below college degree,FT_x_benefits,bucketed_education_Bachelor's Degree,"bucketed_education_Vocational, Associate's Degree, or Certificate",bachelors_x_leadershipexperience,normalized_profile_length
belowcollege_x_workexperience,1.000000,-0.050108,0.880479,-0.066416,-0.260953,-0.073790,-0.150903,0.057029
normalized_benefits_length,-0.050108,1.000000,-0.028916,0.856846,0.034458,0.057783,0.087305,0.200624
bucketed_education_Below college degree,0.880479,-0.028916,1.000000,-0.049250,-0.296376,-0.083807,-0.171387,0.064593
FT_x_benefits,-0.066416,0.856846,-0.049250,1.000000,0.082899,0.052417,0.062655,0.161357
bucketed_education_Bachelor's Degree,-0.260953,0.034458,-0.296376,0.082899,1.000000,-0.120869,0.578277,-0.050340
"bucketed_education_Vocational, Associate's Degree, or Certificate",-0.073790,0.057783,-0.083807,0.052417,-0.120869,1.000000,-0.069896,0.072056
bachelors_x_leadershipexperience,-0.150903,0.087305,-0.171387,0.062655,0.578277,-0.069896,1.000000,0.086006
normalized_profile_length,0.057029,0.200624,0.064593,0.161357,-0.050340,0.072056,0.086006,1.000000


**XGBoost**

In [31]:
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
import re
print("=" * 60)
print("METHOD 1: XGBoost Feature Selection")
print("=" * 60)

# Prepare data for XGBoost (needs binary outcome)
# The outcome is in df, predictors are in df_final
y = y_train  # Binary outcome (0 or 1)
Xf = X_train  # All predictor variables

# Train XGBoost model
xgb_model = xgb.XGBClassifier(
    random_state=42,
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    eval_metric='logloss'
)
xgb_model.fit(Xf, y)

# Get feature importances
xgb_importances = pd.DataFrame({
    'feature': Xf.columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nXGBoost Feature Importances:")
print(xgb_importances)

# Select top N features (e.g., top 10 or features with importance > threshold)
top_n = 10  # Adjust based on your needs
selected_features_xgb = xgb_importances.head(top_n)['feature'].tolist()

print(f"\nTop {top_n} features selected by XGBoost:")
print(selected_features_xgb)

# Fit binomial regression with selected features
X_selected_xgb = X_train[selected_features_xgb]
binomial_xgb = sm.GLM(
    y_train,
    X_selected_xgb,
    family=sm.families.Binomial()
).fit()

print("\n" + "=" * 60)
print("Binomial Regression with XGBoost-Selected Features")
print("=" * 60)
print(binomial_xgb.summary())


METHOD 1: XGBoost Feature Selection

XGBoost Feature Importances:
                                              feature  importance
5                bucketed_education_Bachelor's Degree    0.111039
30                          normalized_profile_length    0.062343
8           bucketed_experience_Leadership experience    0.056384
28                                   has_company_logo    0.055231
17  bucketed_industry_Engineering/Manufacturing/Tr...    0.044236
2                                   salary_bucket_low    0.043255
40                      belowcollege_x_workexperience    0.037837
6             bucketed_education_Below college degree    0.037568
27                       bucketed_industry_Technology    0.037022
51                                    temp_x_benefits    0.028769
3                                salary_bucket_medium    0.026052
43                                      FT_x_benefits    0.024796
7   bucketed_education_Vocational, Associate's Deg...    0.024010
29        

**Combination Scenarios (for Sensitivity Analyis) Predicting Fraud**

In [32]:
baseline_fraudulent_scenario = {
    'salary_bucket': 'missing',
    #'bucketed_education': 'Below college degree',
    'bucketed_experience': 'No experience required',
    'bucketed_employment_type': 'Temporary',
    #'bucketed_industry': 'Engineering/Manufacturing/Trades',
    #'telecommuting': 0, # Removed as 'telecommuting' was not a feature in the model
    'has_company_logo': 0,
    'has_questions': 0,
    #'normalized_profile_length': -1.0,
    #'normalized_description_length': -1.0,
    'normalized_requirements_length': -1.0,
    'normalized_benefits_length': 1.0
}

non_fraudulent_scenario = {
    'salary_bucket': 'high',
    #'bucketed_education': 'Bachelor\'s Degree',
    #'bucketed_experience': 'Leadership experience',
    'bucketed_employment_type': 'Full-time',
    #'bucketed_industry': 'Technology',
    #'telecommuting': 1, # Removed as 'telecommuting' was not a feature in the model
    'has_company_logo': 1,
    'has_questions': 1,
    #'normalized_profile_length': 1.0,
    #'normalized_description_length': 1.0,
    'normalized_requirements_length': 1.0,
    'normalized_benefits_length': 1.0
}

mixed_characteristics_scenario = {
    'salary_bucket': 'missing',
    #'bucketed_education': 'Master\'s Degree',
    #'bucketed_experience': 'Work experience',
    'bucketed_employment_type': 'Full-time',
    #'bucketed_industry': 'Finance',
    #'telecommuting': 0, # Removed as 'telecommuting' was not a feature in the model
    'has_company_logo': 1,
    'has_questions': 1,
    #'normalized_profile_length': 0.0,
    #'normalized_description_length': 0.0,
    'normalized_requirements_length': 0.0,
    'normalized_benefits_length': 0.0
}

print("Three test scenarios (baseline_fraudulent_scenario, non_fraudulent_scenario, mixed_characteristics_scenario) have been defined.")

Three test scenarios (baseline_fraudulent_scenario, non_fraudulent_scenario, mixed_characteristics_scenario) have been defined.


In [33]:
def create_scenario_df(scenario_dict, model_columns):
    """Creates a DataFrame row for a given scenario, matching the model's input structure."""
    # Start with a row of zeros for all model columns
    scenario_df = pd.DataFrame(0, index=[0], columns=model_columns)

    # Set constant
    if 'const' in scenario_df.columns:
        scenario_df['const'] = 1.0

    for key, value in scenario_dict.items():
        # Handle numerical features directly
        if key in ['has_company_logo', 'has_questions',
                   'normalized_profile_length', 'normalized_description_length',
                   'normalized_requirements_length', 'normalized_benefits_length']:
            if key in scenario_df.columns:
                scenario_df[key] = value
        # Handle categorical features (one-hot encoded)
        else:
            # Clean the value for dummy column creation
            cleaned_value = re.sub(r'[<>/\\|?"*:\s]+', '_', str(value)).strip('_')
            dummy_col_name = f"{key}_{cleaned_value}"

            # Check if this dummy column exists in the model's feature set
            if dummy_col_name in scenario_df.columns:
                scenario_df[dummy_col_name] = 1
            # If the column doesn't exist, it might be the base category (encoded as all zeros)

    # NOW ADD INTERACTION TERMS
    # Education x Experience interactions
    if 'bucketed_education' in scenario_dict and 'bucketed_experience' in scenario_dict:
        edu = scenario_dict['bucketed_education']
        exp = scenario_dict['bucketed_experience']

        # Create all education x experience interaction terms
        edu_clean = re.sub(r'[<>/\\|?"*:\s]+', '_', str(edu)).strip('_')
        exp_clean = re.sub(r'[<>/\\|?"*:\s]+', '_', str(exp)).strip('_')

        # Map education to short form
        edu_map = {
            "Above_Bachelor_s": "abovebachelor",
            "Bachelor_s_Degree": "bachelors",
            "Below_college_degree": "belowcollege",
            "Vocational_Associate_s_Degree_or_Certificate": "vocational"
        }

        # Map experience to short form
        exp_map = {
            "Leadership_experience": "leadershipexperience",
            "Work_experience": "workexperience"
        }

        edu_short = edu_map.get(edu_clean, edu_clean.lower())
        exp_short = exp_map.get(exp_clean, exp_clean.lower())

        interaction_col = f"{edu_short}_x_{exp_short}"
        if interaction_col in scenario_df.columns:
            scenario_df[interaction_col] = 1

    # Employment type x length interactions
    if 'bucketed_employment_type' in scenario_dict:
        emp_type = scenario_dict['bucketed_employment_type']

        # Map employment type to short form
        emp_map = {
            "Full-time": "FT",
            "Part-time": "PT",
            "Temporary": "temp"
        }

        emp_short = emp_map.get(emp_type, emp_type)

        # Create interactions with all length variables
        for length_var in ['profile', 'benefits', 'desc', 'req']:
            interaction_col = f"{emp_short}_x_{length_var}"
            if interaction_col in scenario_df.columns:
                # Get the corresponding normalized length value
                length_map = {
                    'profile': 'normalized_profile_length',
                    'benefits': 'normalized_benefits_length',
                    'desc': 'normalized_description_length',
                    'req': 'normalized_requirements_length'
                }
                length_feature = length_map[length_var]
                if length_feature in scenario_dict:
                    scenario_df[interaction_col] = scenario_dict[length_feature]

    return scenario_df


def calculate_actual_fraud_rate(scenario_dict, original_df):
    """Calculates the actual fraud rate for a given scenario from the original DataFrame.

    Note: This filters on categorical features only. Normalized numerical features
    cannot be matched back to original values.
    """
    filtered_df = original_df.copy()

    # Only filter on categorical features that exist in the original dataframe
    categorical_features = ['salary_bucket', 'bucketed_education', 'bucketed_experience',
                           'bucketed_employment_type', 'bucketed_industry']

    for key, value in scenario_dict.items():
        if key in categorical_features and key in filtered_df.columns:
            filtered_df = filtered_df[filtered_df[key] == value]

    if not filtered_df.empty:
        actual_rate = filtered_df['fraudulent'].mean()
        count = len(filtered_df)
        return actual_rate, count
    else:
        return np.nan, 0


print("Updated helper functions defined.")

Updated helper functions defined.


In [34]:
import numpy as np

scenarios = {
    'Baseline Fraudulent Job Posting': baseline_fraudulent_scenario,
    'Typical Non-Fraudulent Job Posting': non_fraudulent_scenario,
    'Mixed Characteristics Job Posting': mixed_characteristics_scenario
}

# Get the actual column names from X_train (the data the model was trained on)
df_final_cols = X_train.columns.tolist()

print("\n--- Fraud Prediction and Actual Rate Comparison ---")
print(f"Model expects {len(df_final_cols)} features")

for name, scenario_dict in scenarios.items():
    print(f"\n{'='*60}")
    print(f"Scenario: {name}")
    print(f"{'='*60}")

    # Display scenario details
    print("\nScenario Characteristics:")
    for key, value in scenario_dict.items():
        print(f"  {key}: {value}")

    # 1. Create DataFrame for prediction
    scenario_df_for_pred = create_scenario_df(scenario_dict, df_final_cols)

    # Verify all columns are present
    missing_cols = set(df_final_cols) - set(scenario_df_for_pred.columns)
    if missing_cols:
        print(f"\nWarning: Missing columns: {missing_cols}")
        for col in missing_cols:
            scenario_df_for_pred[col] = 0

    # Ensure column order matches model
    scenario_df_for_pred = scenario_df_for_pred[df_final_cols]

    # 2. Predict fraud probability
    predicted_fraud_prob = result.predict(scenario_df_for_pred)[0]
    print(f"\n Predicted Probability of Fraud: {predicted_fraud_prob:.4f} ({predicted_fraud_prob*100:.2f}%)")

    # 3. Calculate actual fraud rate
    actual_fraud_rate, sample_count = calculate_actual_fraud_rate(scenario_dict, df)
    if not np.isnan(actual_fraud_rate):
        print(f" Actual Observed Fraud Rate: {actual_fraud_rate:.4f} ({actual_fraud_rate*100:.2f}%)")
        print(f"   (Based on {sample_count} matching job postings)")
    else:
        print(" Actual Observed Fraud Rate: No matching job postings found")

    # Show interpretation
    #if predicted_fraud_prob > 0.5:
       # print(f"  Model classifies this as: Fraudulent")
   # else:
      #  print(f"  Model classifies this as: Legitimate")


--- Fraud Prediction and Actual Rate Comparison ---
Model expects 54 features

Scenario: Baseline Fraudulent Job Posting

Scenario Characteristics:
  salary_bucket: missing
  bucketed_experience: No experience required
  bucketed_employment_type: Temporary
  has_company_logo: 0
  has_questions: 0
  normalized_requirements_length: -1.0
  normalized_benefits_length: 1.0

 Predicted Probability of Fraud: 0.0080 (0.80%)
 Actual Observed Fraud Rate: 0.0224 (2.24%)
   (Based on 715 matching job postings)

Scenario: Typical Non-Fraudulent Job Posting

Scenario Characteristics:
  salary_bucket: high
  bucketed_employment_type: Full-time
  has_company_logo: 1
  has_questions: 1
  normalized_requirements_length: 1.0
  normalized_benefits_length: 1.0

 Predicted Probability of Fraud: 0.0134 (1.34%)
 Actual Observed Fraud Rate: 0.0692 (6.92%)
   (Based on 130 matching job postings)

Scenario: Mixed Characteristics Job Posting

Scenario Characteristics:
  salary_bucket: missing
  bucketed_employme

In [35]:
def create_scenario_df(scenario_dict, model_columns):
    """Creates a DataFrame row for a given scenario, matching the model's input structure."""
    # Start with a row of zeros for all model columns
    scenario_df = pd.DataFrame(0, index=[0], columns=model_columns)

    # Set constant
    if 'const' in scenario_df.columns:
        scenario_df['const'] = 1.0

    for key, value in scenario_dict.items():
        # Handle numerical features directly
        if key in ['has_company_logo', 'has_questions',
                   'normalized_profile_length', 'normalized_description_length',
                   'normalized_requirements_length', 'normalized_benefits_length']:
            if key in scenario_df.columns:
                scenario_df[key] = value
        # Handle categorical features (one-hot encoded)
        else:
            # Clean the value for dummy column creation
            cleaned_value = re.sub(r'[<>/\\|?"*:\s]+', '_', str(value)).strip('_')
            dummy_col_name = f"{key}_{cleaned_value}"

            # Check if this dummy column exists in the model's feature set
            if dummy_col_name in scenario_df.columns:
                scenario_df[dummy_col_name] = 1
            # If the column doesn't exist, it might be the base category (encoded as all zeros)

    # NOW ADD INTERACTION TERMS
    # Education x Experience interactions
    if 'bucketed_education' in scenario_dict and 'bucketed_experience' in scenario_dict:
        edu = scenario_dict['bucketed_education']
        exp = scenario_dict['bucketed_experience']

        # Create all education x experience interaction terms
        edu_clean = re.sub(r'[<>/\\|?"*:\s]+', '_', str(edu)).strip('_')
        exp_clean = re.sub(r'[<>/\\|?"*:\s]+', '_', str(exp)).strip('_')

        # Map education to short form
        edu_map = {
            "Above_Bachelor_s": "abovebachelor",
            "Bachelor_s_Degree": "bachelors",
            "Below_college_degree": "belowcollege",
            "Vocational_Associate_s_Degree_or_Certificate": "vocational"
        }

        # Map experience to short form
        exp_map = {
            "Leadership_experience": "leadershipexperience",
            "Work_experience": "workexperience"
        }

        edu_short = edu_map.get(edu_clean, edu_clean.lower())
        exp_short = exp_map.get(exp_clean, exp_clean.lower())

        interaction_col = f"{edu_short}_x_{exp_short}"
        if interaction_col in scenario_df.columns:
            scenario_df[interaction_col] = 1

    # Employment type x length interactions
    if 'bucketed_employment_type' in scenario_dict:
        emp_type = scenario_dict['bucketed_employment_type']

        # Map employment type to short form
        emp_map = {
            "Full-time": "FT",
            "Part-time": "PT",
            "Temporary": "temp"
        }

        emp_short = emp_map.get(emp_type, emp_type)

        # Create interactions with all length variables
        for length_var in ['profile', 'benefits', 'desc', 'req']:
            interaction_col = f"{emp_short}_x_{length_var}"
            if interaction_col in scenario_df.columns:
                # Get the corresponding normalized length value
                length_map = {
                    'profile': 'normalized_profile_length',
                    'benefits': 'normalized_benefits_length',
                    'desc': 'normalized_description_length',
                    'req': 'normalized_requirements_length'
                }
                length_feature = length_map[length_var]
                if length_feature in scenario_dict:
                    scenario_df[interaction_col] = scenario_dict[length_feature]

    return scenario_df


def calculate_actual_fraud_rate(scenario_dict, original_df):
    """Calculates the actual fraud rate for a given scenario from the original DataFrame.

    Note: This filters on categorical features only. Normalized numerical features
    cannot be matched back to original values.
    """
    filtered_df = original_df.copy()

    # Only filter on categorical features that exist in the original dataframe
    categorical_features = ['salary_bucket', 'bucketed_education', 'bucketed_experience',
                           'bucketed_employment_type', 'bucketed_industry']

    for key, value in scenario_dict.items():
        if key in categorical_features and key in filtered_df.columns:
            filtered_df = filtered_df[filtered_df[key] == value]

    if not filtered_df.empty:
        actual_rate = filtered_df['fraudulent'].mean()
        count = len(filtered_df)
        return actual_rate, count
    else:
        return np.nan, 0


print("Updated helper functions defined.")

Updated helper functions defined.


In [36]:
from scipy.special import expit
import pandas as pd
import numpy as np

# Use the final trained model (result) instead of binomial_model or interaction_model
final_model = result

# Extract model parameters
b0 = final_model.params["const"]  # Intercept

# Get all salary bucket columns from the final model
salary_columns = [col for col in final_model.params.index if col.startswith('salary_bucket_')]

print("=" * 70)
print("FINAL MODEL INTERPRETATION: SALARY BRACKET EFFECT")
print("=" * 70)

print(f"\nModel Parameters:")
print(f"  Intercept (b0): {b0:.4f}")
print(f"\n  Salary Bucket Coefficients:")

# Display all salary bucket coefficients
for col in salary_columns:
    coef = final_model.params[col]
    print(f"    {col}: {coef:.4f}")

# Note: In your final model, the base category is 'salary_bucket_missing'
# So all other categories are compared to missing salary information

print(f"\n{'='*70}")
print("PROBABILITY CALCULATIONS FOR EACH SALARY BRACKET")
print(f"{'='*70}")

# Calculate probabilities for each salary bracket
# When salary is missing (base category, all salary indicators = 0)
p_missing = expit(b0)
print(f"\nBase Category (missing salary):")
print(f"  P(fraudulent | salary missing): {p_missing:.4f} ({p_missing*100:.2f}%)")

# Calculate for each non-missing salary bracket
salary_probs = []
for col in salary_columns:
    coef = final_model.params[col]
    p_bracket = expit(b0 + coef)
    odds_ratio = np.exp(coef)

    # Get actual data mean if available
    if col in X_train.columns:
        bracket_mean = X_train[col].mean()
    else:
        bracket_mean = np.nan

    salary_probs.append({
        'bracket': col.replace('salary_bucket_', ''),
        'coefficient': coef,
        'probability': p_bracket,
        'odds_ratio': odds_ratio,
        'diff_from_missing': p_bracket - p_missing,
        'proportion_in_data': bracket_mean
    })

    print(f"\n{col.replace('salary_bucket_', '')} salary bracket:")
    print(f"  Coefficient: {coef:.4f}")
    print(f"  P(fraudulent): {p_bracket:.4f} ({p_bracket*100:.2f}%)")
    print(f"  Odds Ratio vs. missing: {odds_ratio:.4f}")
    print(f"  Difference from missing: {(p_bracket - p_missing):.4f} ({(p_bracket - p_missing)*100:.2f} pp)")
    if not np.isnan(bracket_mean):
        print(f"  Proportion in training data: {bracket_mean:.4f} ({bracket_mean*100:.2f}%)")

print(f"\n{'='*70}")
print("SALARY BRACKETS RANKED BY FRAUD RISK")
print(f"{'='*70}")

# Create DataFrame and sort by probability
salary_df = pd.DataFrame(salary_probs).sort_values('probability', ascending=False)

print("\n" + salary_df.to_string(index=False))

print(f"\n{'='*70}")
print("KEY INSIGHTS")
print(f"{'='*70}")

# Find highest and lowest risk brackets
highest_risk = salary_df.iloc[0]
lowest_risk = salary_df.iloc[-1]

print(f"\nHighest Fraud Risk: {highest_risk['bracket']}")
print(f"  P(fraud) = {highest_risk['probability']:.4f} ({highest_risk['probability']*100:.2f}%)")

print(f"\nLowest Fraud Risk: {lowest_risk['bracket']}")
print(f"  P(fraud) = {lowest_risk['probability']:.4f} ({lowest_risk['probability']*100:.2f}%)")

print(f"\nRisk Range:")
print(f"  Absolute difference: {(highest_risk['probability'] - lowest_risk['probability']):.4f}")
print(f"  ({(highest_risk['probability'] - lowest_risk['probability'])*100:.2f} percentage points)")

# Compare specific brackets if they exist
print(f"\n{'='*70}")
print("SPECIFIC COMPARISONS")
print(f"{'='*70}")

# Low vs Medium
if 'low' in salary_df['bracket'].values and 'medium' in salary_df['bracket'].values:
    low_data = salary_df[salary_df['bracket'] == 'low'].iloc[0]
    medium_data = salary_df[salary_df['bracket'] == 'medium'].iloc[0]

    print(f"\nLow Salary (<$50k) vs Medium Salary ($50k-$100k):")
    print(f"  Low:    P(fraud) = {low_data['probability']:.4f} ({low_data['probability']*100:.2f}%)")
    print(f"  Medium: P(fraud) = {medium_data['probability']:.4f} ({medium_data['probability']*100:.2f}%)")
    print(f"  Difference: {(low_data['probability'] - medium_data['probability']):.4f}")
    print(f"  ({abs(low_data['probability'] - medium_data['probability'])*100:.2f} pp)")

    if low_data['probability'] > medium_data['probability']:
        print(f"  → Low salary jobs are MORE risky by {(low_data['probability'] - medium_data['probability'])*100:.2f} pp")
    else:
        print(f"  → Medium salary jobs are MORE risky by {(medium_data['probability'] - low_data['probability'])*100:.2f} pp")

# High vs Medium
if 'high' in salary_df['bracket'].values and 'medium' in salary_df['bracket'].values:
    high_data = salary_df[salary_df['bracket'] == 'high'].iloc[0]
    medium_data = salary_df[salary_df['bracket'] == 'medium'].iloc[0]

    print(f"\nHigh Salary (>$100k) vs Medium Salary ($50k-$100k):")
    print(f"  High:   P(fraud) = {high_data['probability']:.4f} ({high_data['probability']*100:.2f}%)")
    print(f"  Medium: P(fraud) = {medium_data['probability']:.4f} ({medium_data['probability']*100:.2f}%)")
    print(f"  Difference: {(high_data['probability'] - medium_data['probability']):.4f}")
    print(f"  ({abs(high_data['probability'] - medium_data['probability'])*100:.2f} pp)")

    if high_data['probability'] > medium_data['probability']:
        print(f"  → High salary jobs are MORE risky by {(high_data['probability'] - medium_data['probability'])*100:.2f} pp")
    else:
        print(f"  → Medium salary jobs are MORE risky by {(medium_data['probability'] - high_data['probability'])*100:.2f} pp")

print(f"\n{'='*70}")
print("INTERPRETATION NOTES")
print(f"{'='*70}")
print("""
1. All comparisons are relative to jobs with MISSING salary information
2. The model controls for education, experience, employment type, industry,
   and various job posting characteristics (logo, questions, text lengths)
3. These are MARGINAL effects - holding all other variables constant
4. Coefficients show the log-odds change; odds ratios show multiplicative effects
""")

print("=" * 70)

FINAL MODEL INTERPRETATION: SALARY BRACKET EFFECT

Model Parameters:
  Intercept (b0): -2.8719

  Salary Bucket Coefficients:
    salary_bucket_high: -0.4177
    salary_bucket_low: 1.4181
    salary_bucket_medium: -0.3104

PROBABILITY CALCULATIONS FOR EACH SALARY BRACKET

Base Category (missing salary):
  P(fraudulent | salary missing): 0.0536 (5.36%)

high salary bracket:
  Coefficient: -0.4177
  P(fraudulent): 0.0359 (3.59%)
  Odds Ratio vs. missing: 0.6585
  Difference from missing: -0.0176 (-1.76 pp)
  Proportion in training data: 0.0136 (1.36%)

low salary bracket:
  Coefficient: 1.4181
  P(fraudulent): 0.1894 (18.94%)
  Odds Ratio vs. missing: 4.1294
  Difference from missing: 0.1359 (13.59 pp)
  Proportion in training data: 0.0745 (7.45%)

medium salary bracket:
  Coefficient: -0.3104
  P(fraudulent): 0.0398 (3.98%)
  Odds Ratio vs. missing: 0.7332
  Difference from missing: -0.0137 (-1.37 pp)
  Proportion in training data: 0.0434 (4.34%)

SALARY BRACKETS RANKED BY FRAUD RISK

b

In [37]:
print("\n" + "=" * 70)
print("INDUSTRY EFFECTS ON FRAUD RISK")
print("=" * 70)

# Get all industry columns
industry_columns = [col for col in final_model.params.index if col.startswith('bucketed_industry_')]

industry_probs = []
for col in industry_columns:
    coef = final_model.params[col]
    p_industry = expit(b0 + coef)
    odds_ratio = np.exp(coef)

    if col in X_train.columns:
        industry_mean = X_train[col].mean()
    else:
        industry_mean = np.nan

    industry_probs.append({
        'industry': col.replace('bucketed_industry_', ''),
        'coefficient': coef,
        'probability': p_industry,
        'odds_ratio': odds_ratio,
        'proportion_in_data': industry_mean
    })

industry_df = pd.DataFrame(industry_probs).sort_values('probability', ascending=False)

print("\nIndustries Ranked by Fraud Risk:")
print(industry_df.to_string(index=False))

print(f"\n{'='*70}")
print("TOP 3 RISKIEST INDUSTRIES")
print(f"{'='*70}")
for i, row in industry_df.head(3).iterrows():
    print(f"\n{i+1}. {row['industry']}")
    print(f"   P(fraud) = {row['probability']:.4f} ({row['probability']*100:.2f}%)")
    print(f"   Odds Ratio = {row['odds_ratio']:.4f}")
    if not np.isnan(row['proportion_in_data']):
        print(f"   {row['proportion_in_data']*100:.2f}% of jobs in training data")

print(f"\n{'='*70}")
print("TOP 3 SAFEST INDUSTRIES")
print(f"{'='*70}")
for i, row in industry_df.tail(3).iloc[::-1].iterrows():
    print(f"\n{i+1}. {row['industry']}")
    print(f"   P(fraud) = {row['probability']:.4f} ({row['probability']*100:.2f}%)")
    print(f"   Odds Ratio = {row['odds_ratio']:.4f}")
    if not np.isnan(row['proportion_in_data']):
        print(f"   {row['proportion_in_data']*100:.2f}% of jobs in training data")

print("=" * 70)


INDUSTRY EFFECTS ON FRAUD RISK

Industries Ranked by Fraud Risk:
                        industry  coefficient  probability  odds_ratio  proportion_in_data
Engineering/Manufacturing/Trades     1.964333     0.287502    7.130153            0.046836
                     Real Estate     0.871733     0.119187    2.391050            0.027875
                         Finance     0.630958     0.096135    1.879411            0.089151
                  Leisure/Luxury     0.552717     0.089548    1.737968            0.039804
                           Other     0.473128     0.083268    1.605007            0.041436
          Consulting/development     0.454361     0.081846    1.575167            0.015445
                       Art/Media     0.385161     0.076794    1.469851            0.018458
       Marketing, PR, Recruiting     0.301471     0.071067    1.351846            0.046836
                      Operations     0.218036     0.065752    1.243632            0.012557
                      Te